# Online Retail II — SQL-часть

Повторяю ключевые расчёты из основного ноутбука `retail.ipynb` на SQL, чтобы сверить результаты двумя способами.

**Данные:** таблица `sales` (реальные продажи: цена > 0, без возвратов и бухгалтерских записей, только товарные коды), выгружена из основного ноутбука в базу SQLite `retail.db`.

**Содержание:**
- Проверка загрузки
1. Выручка по каналам
2. Выручка по месяцам
3. Топ-10 товаров
4. RFM-сегментация: таблица клиентов → баллы → сегменты → итог

**Что используется:** `group by`, подзапросы, `case when`, работа с датами (`strftime`, `julianday`), CTE (`with`), оконные функции (`ntile`, `sum() over ()`).

In [1]:
import pandas as pd
import sqlite3
conn = sqlite3.connect('retail.db')   # подключаемся к той же базе

## Проверка загрузки

In [2]:
query = """
select count(*) as rows_count,
       sum(Revenue) as total_revenue
from sales
"""
pd.read_sql(query, conn)

,rows_count,total_revenue
0,1003214,19642692.15


1 003 214 строк и 19 642 692 выручки — совпадает с таблицей `sales` в основном ноутбуке, загрузка прошла без потерь.

## 1. Выручка по каналам

Опт — клиенты с ID, розница — покупатели без ID. Доля считается через подзапрос с общей выручкой.

In [3]:
# запросим: 
# 1. Сумму выручки
# 2. Долю от всей выручки в %
query = """
select  case when "Customer ID null" = 1 then 'розница' else 'опт' end as channel,
        round(sum(Revenue), 0) as sum_revenue,
        round(sum(Revenue) / (select sum(Revenue) from sales) * 100, 1) as share_revenue
from sales
group by "Customer ID null"
"""
pd.read_sql(query, conn)

,channel,sum_revenue,share_revenue
0,опт,17068568.0,86.9
1,розница,2574124.0,13.1


**Вывод:** опт 86,9% (17,07 млн), розница 13,1% (2,57 млн) — до копейки совпадает с pandas.

## 2. Выручка по месяцам

Месяц вырезается из даты через `strftime('%Y-%m', ...)`.

In [4]:
# Выручка по месяцам
query = """
select  strftime('%Y-%m', InvoiceDate) as month,
        sum(Revenue) as revenue_sum
from sales
group by month
order by month
"""
pd.read_sql(query, conn)

,month,revenue_sum
0,2009-12,797738.02
1,2010-01,612271.20
2,2010-02,537893.54
3,2010-03,761665.62
4,2010-04,646233.18
5,2010-05,643474.28
6,2010-06,696439.47
7,2010-07,632983.51
8,2010-08,674036.87
9,2010-09,869224.44


**Выводы по месяцам**

1. **Сезонность подтверждается в деньгах.** Ноябрь — пик в оба года: 1,43 млн в 2010 и 1,45 млн в 2011, примерно вдвое больше, чем в августе (0,67 и 0,72 млн). Рост начинается в сентябре. Это совпадает с когортным анализом: клиенты чаще всего возвращаются в ноябре.
2. **Бизнес почти не растёт.** Январь–ноябрь 2011 против тех же месяцев 2010 — всего +3%. Сезон повторяется, а роста год к году почти нет.
3. **Ограничение.** Декабрь 2011 (614 тыс.) — данные за 9 рабочих дней.

## 3. Топ-10 товаров по выручке

Группировка только по `StockCode`: у одного кода бывает несколько вариантов описания, поэтому название берётся через `max(Description)`. Число заказов `count(distinct Invoice)` показывает, из скольких покупок сложилась выручка.

In [5]:
# Топ-10 товаров по выручке
query = """
select  
        "StockCode",
        max(Description) as Description,
        count(distinct Invoice) as orders,
        sum(Quantity) as total_qty,
        sum(Revenue) as total_revenue
from sales
group by "StockCode" 
order by "total_revenue" desc 
limit 10
"""
pd.read_sql(query, conn)

,StockCode,Description,orders,total_qty,total_revenue
0,22423,REGENCY CAKESTAND 3 TIER,3918,26478,330590.32
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,5365,94203,257724.71
2,85099B,RED RETROSPOT JUMBO BAG,3989,96757,180569.34
3,23843,"PAPER CRAFT , LITTLE BIRDIE",1,80995,168469.60
4,47566,PARTY BUNTING,2674,28200,148318.28
5,84879,ASSORTED COLOUR BIRD ORNAMENT,2807,80082,129324.49
6,22086,PAPER CHAIN KIT 50'S CHRISTMAS,2018,35084,117760.29
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,247,78033,81700.92
8,79321,CHILLI LIGHTS,1135,15841,80540.88
9,22197,SMALL POPCORN HOLDER,2322,88499,79520.20


**Выводы по товарам**

1. **Фальшивые лидеры.** PAPER CRAFT, LITTLE BIRDIE — всего 1 заказ: вся выручка товара, 168 тыс., пришла из одной покупки на 80 995 штук, которую потом отменили. MEDIUM CERAMIC TOP STORAGE JAR — реальный товар, 247 заказов, но в топ его вытащил один отменённый заказ на 74 215 штук. Без него у товара около 4,5 тыс. выручки. Поэтому рядом с суммой всегда стоит смотреть число заказов: настоящий хит набирает выручку тысячами покупок, а не одной.
2. **Концентрация.** Топ-10 товаров дают около 8% выручки (без двух фальшивых лидеров — около 7%), а среди клиентов 25% дают 69%. Ассортимент широкий и без явных хитов: потеря любого одного товара почти не заметна. Главный риск магазина — не товары, а клиенты: уход нескольких крупных оптовиков ударит сильнее, чем исчезновение любого товара. Поэтому усилия стоит направлять на удержание ключевых клиентов, а не на продвижение отдельных позиций.
3. **Ограничение.** Это валовая выручка: отмены лежат в счетах на C, которых нет в таблице `sales`, поэтому отменённые заказы в ней остаются.

## 4. RFM-сегментация на SQL

### Шаг 1. Таблица клиентов

Одна строка на клиента с ID:
- **recency** — дней от последней покупки до даты «снимка» (последняя дата в данных + 1 день), через `julianday`;
- **frequency** — число покупок (уникальных счетов);
- **monetary** — сумма выручки.

In [6]:
query = """
select  "Customer ID",
        cast(
            julianday((select datetime(max(InvoiceDate), '+1 day') from sales))
            - julianday(max(InvoiceDate)) 
        as integer) as recency,
        count(distinct Invoice) as frequency,
        sum(Revenue) as monetary
from sales
where "Customer ID null" = 0
group by "Customer ID"
"""
pd.read_sql(query, conn)

,Customer ID,recency,frequency,monetary
0,12346.0,326,3,77352.96
1,12347.0,2,8,4921.53
2,12348.0,75,5,1658.40
3,12349.0,19,3,3678.69
4,12350.0,310,1,294.40
...,...,...,...,...
5847,18283.0,4,22,2658.95
5848,18284.0,432,1,411.68
5849,18285.0,661,1,377.00
5850,18286.0,477,2,1246.43


**Проверка:** 5 852 клиента, как в pandas. Клиент 12346: recency 326, frequency 3, monetary 77 352,96 — совпадает.

### Шаг 2. Присваиваем баллы 1–5 через `ntile(5)`

`ntile(5)` выстраивает клиентов по показателю и делит на 5 равных групп. Балл 1 получают те, кто в начале очереди, поэтому худших ставим в начало: recency — по убыванию (давно не покупал = плохо), frequency и monetary — по возрастанию.

**Ничьи:** у многих клиентов одинаковое число покупок, и граница между баллами проходит внутри таких групп. Правило задано явно: при равной частоте выше балл получает тот, кто принёс больше денег (`order by frequency, monetary`).

In [7]:
query = """
with customers as (
select  "Customer ID",
        cast(
            julianday((select datetime(max(InvoiceDate), '+1 day') from sales))
            - julianday(max(InvoiceDate)) 
        as integer) as recency,
        count(distinct Invoice) as frequency,
        sum(Revenue) as monetary
from sales
where "Customer ID null" = 0
group by "Customer ID"
)
select  *,
        ntile(5) over (order by recency desc)  as r,
        ntile(5) over (order by frequency, monetary)  as f,
        ntile(5) over (order by monetary)  as m
from customers
order by "Customer ID"
"""
pd.read_sql(query, conn)

,Customer ID,recency,frequency,monetary,r,f,m
0,12346.0,326,3,77352.96,2,3,5
1,12347.0,2,8,4921.53,5,5,5
2,12348.0,75,5,1658.40,3,4,4
3,12349.0,19,3,3678.69,5,3,5
4,12350.0,310,1,294.40,2,1,2
...,...,...,...,...,...,...,...
5847,18283.0,4,22,2658.95,5,5,4
5848,18284.0,432,1,411.68,1,2,2
5849,18285.0,661,1,377.00,1,2,2
5850,18286.0,477,2,1246.43,1,3,4


**Проверка:** у клиента 12346 баллы R=2, F=3, M=5 — как в pandas. У части клиентов на границах баллов F отличается от pandas (например, 12347: 4 в pandas, 5 здесь) — это разное правило для ничьих, см. выводы ниже.

### Шаг 3. Сегменты через `case when`

Правила проверяются сверху вниз, срабатывает первое подходящее — как в функции `get_segment` в основном ноутбуке. Баллы `r` и `f` создаются в слое `scores`, поэтому сегменты считаются уже поверх него: CTE идут цепочкой `customers → scores → …`.

In [8]:
query = """
with customers as (
select  "Customer ID",
        cast(
            julianday((select datetime(max(InvoiceDate), '+1 day') from sales))
            - julianday(max(InvoiceDate)) 
        as integer) as recency,
        count(distinct Invoice) as frequency,
        sum(Revenue) as monetary
from sales
where "Customer ID null" = 0
group by "Customer ID"
),
scores as (
select  *,
        ntile(5) over (order by recency desc)  as r,
        ntile(5) over (order by frequency, monetary)  as f,
        ntile(5) over (order by monetary)  as m
from customers
)
select  *,
        case 
        when r >= 4 and f >= 4 then 'Champions'
        when r >= 3 and f >= 3 then  'Loyal'
        when r >= 4 then 'New'
        when r <= 2 and f >= 3 then 'At_Risk'
        when r = 2 or r = 3 then 'Hibernating'
        else 'Lost'
        end as segment
from scores
""" 
pd.read_sql(query, conn)

,Customer ID,recency,frequency,monetary,r,f,m,segment
0,17592.0,739,1,148.30,1,1,1,Lost
1,15833.0,738,1,80.40,1,1,1,Lost
2,17818.0,738,1,124.78,1,1,1,Lost
3,17056.0,738,1,128.60,1,1,1,Lost
4,17087.0,738,1,221.53,1,1,1,Lost
...,...,...,...,...,...,...,...,...
5847,18102.0,1,145,580987.04,5,5,5,Champions
5848,14606.0,1,184,29648.75,5,5,5,Champions
5849,15311.0,1,207,114671.42,5,5,5,Champions
5850,12748.0,1,322,49173.79,5,5,5,Champions


### Шаг 4. Итог: клиенты и выручка по сегментам

Доля клиентов — через оконную функцию `sum(count(*)) over ()` (сумма по всем строкам результата), доля выручки — через подзапрос. 

In [9]:
query = """
with customers as (
select  "Customer ID",
        cast(
            julianday((select datetime(max(InvoiceDate), '+1 day') from sales))
            - julianday(max(InvoiceDate)) 
        as integer) as recency,
        count(distinct Invoice) as frequency,
        sum(Revenue) as monetary
from sales
where "Customer ID null" = 0
group by "Customer ID"
),
scores as (
select  *,
        ntile(5) over (order by recency desc)  as r,
        ntile(5) over (order by frequency, monetary)  as f,
        ntile(5) over (order by monetary)  as m
from customers
),
table_segments as (
select  *,
        case 
        when r >= 4 and f >= 4 then 'Champions'
        when r >= 3 and f >= 3 then  'Loyal'
        when r >= 4 then 'New'
        when r <= 2 and f >= 3 then 'At_Risk'
        when r = 2 or r = 3 then 'Hibernating'
        else 'Lost'
        end as segment
from scores
)
select  segment,
        count(*) as clients,
        round(count(*) * 100.0 / sum(count(*)) over (), 1) as clients_share,
        cast(round(sum(monetary), 0) as integer) as revenue,
        round(sum(monetary)* 100.0 / (select sum(monetary) from table_segments), 1) as revenue_share
from table_segments
group by segment
order by revenue desc
""" 
pd.read_sql(query, conn)

,segment,clients,clients_share,revenue,revenue_share
0,Champions,1473,25.2,11884168,69.6
1,Loyal,1221,20.9,2632523,15.4
2,At_Risk,816,13.9,1680641,9.8
3,Hibernating,1002,17.1,388901,2.3
4,Lost,902,15.4,311645,1.8
5,New,438,7.5,170690,1.0


**Сверка с pandas**

| Сегмент | Клиенты: pandas → SQL | Выручка: pandas → SQL |
|---|---|---|
| Champions | 1 476 → 1 473 | 11,82 → 11,88 млн |
| Loyal | 1 207 → 1 221 | 2,42 → 2,63 млн |
| At_Risk | 828 → 816 | 1,59 → 1,68 млн |
| Hibernating | 1 001 → 1 002 | 462 → 389 тыс. |
| Lost | 889 → 902 | 359 → 312 тыс. |
| New | 451 → 438 | 416 → 171 тыс. |

**Выводы**

1. **Главный результат подтверждён.** Champions — четверть клиентов (25,2%) и около 70% выручки (69,6% против 69,2% в pandas). Итоги сходятся: 5 852 клиента и 17 068 568 выручки.
2. **Расхождения — на границах баллов.** Число клиентов в сегментах отличается не больше чем на 14, но деньги у New и Loyal разошлись заметно: у New 416 → 171 тыс. при разнице всего в 13 клиентов. Причина — ничьи по частоте на границе F2/F3. В pandas (`rank(method='first')`) равных клиентов разводил порядок строк, то есть номер клиента. Здесь — сумма денег: богатые клиенты с 2 покупками получают F3 и уходят в Loyal, а не в New.
3. **Вывод о методе.** Для клиентов с одинаковыми значениями сегмент зависит от выбранного правила разведения ничьих, и это двигает деньги между сегментами. Поэтому правило нужно задавать явно и обосновывать. Здесь выбрано «при равной частоте выше тот, кто больше заплатил». На крупные сегменты (Champions, At_Risk) выбор правила почти не влияет, а для New и Loyal цифры стоит считать ориентировочными.
4. **Ловушка SQLite.** Пока правило не было записано явно, результат был тем же: SQLite разводил ничьи в окне `f` по соседнему окну `m`. Полагаться на такое скрытое поведение нельзя — перестановка колонок тихо изменила бы сегменты.

## Итоги SQL-части

- Все ключевые расчёты из pandas воспроизведены на SQL и сверены: каналы (87/13), выручка по месяцам, RFM-сегменты (Champions ≈ 25% клиентов → ≈ 70% выручки).
- Новые находки: в топе товаров есть «фальшивые лидеры» из одного отменённого заказа; товары почти не концентрированы (топ-10 ≈ 8% выручки), в отличие от клиентов; выручка год к году почти не растёт (+3%).
- Ограничения: валовая выручка содержит отменённые заказы; декабрь 2011 неполный; сегменты пограничных клиентов зависят от правила разведения ничьих.